# Data Preparation (Preparação dos dados)

### Biblioteca / Configuração

In [1]:
# Dependências
import sys
#!{sys.executable} -m pip install --disable-pip-version-check -r ../requirements.txt -q
print('Bibliotecas instaladas')

Bibliotecas instaladas


In [2]:
# Acesso aos modulos do diretório
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT)) 

# Manipulação dos dados
import pandas as pd
import numpy as np
import pickle

# Visualização dos dados
import matplotlib.pyplot as plt
import seaborn as sns

# Diretórios
from config.paths import *
from config.function_models import *

# Avisos
import warnings
warnings.filterwarnings('ignore')

# Configuração
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', None)

print('Ambiente Configurado')

Diretórios carregadas com sucesso
Ambiente Configurado


## Parâmetros Globais

In [3]:
# define a coluna alvo do modelo
TARGET = 'FPD'

### Carregamento dos dados 

In [4]:
# lê o arquivo parquet e carrega em um DataFrame
test = pd.read_parquet(RAW_DIR / 'base_test.parquet')
controle = pd.read_parquet(RAW_DIR / 'grupo_controle.parquet')

print(f"Test Linhas: {test.shape[0]:,} | Colunas: {test.shape[1]:,}")
print(f"Controle Linhas: {controle.shape[0]:,} | Colunas: {controle.shape[1]:,}")

Test Linhas: 389,550 | Colunas: 129
Controle Linhas: 60,197 | Colunas: 130


### Artefatos

In [5]:
# Carregar a lista de features do features_stage_02.pkl
with open(ARTIFACT_DIR / 'features_stage_02.pkl', 'rb') as f:
    features_stage_02 = pickle.load(f)

# carregar estatísticas de imputação
with open(Path(ARTIFACT_DIR) / 'stats_nulo.pkl', 'rb') as f:
    stats = pickle.load(f)

### Tratamento Teste e Grupo controle

In [6]:
# Backup dos dados originais
test_01 = test.copy()
controle_01 = controle.copy()

# Carregar lista de features do Feature selection e recolocar a target original
test_01 = test[features_stage_02 + [TARGET]].copy()
controle_01 = controle[features_stage_02 + [TARGET]].copy()

print(f'Teste: {test_01.shape}')
print(f'Controle: {controle_01.shape}')

Teste: (389550, 19)
Controle: (60197, 19)


In [7]:
# Base de teste
test_01 = sanitize_dataframe(test_01, inplace=True)
test_01 = preencher_missing_com_stats(test_01, stats)
# Base controle
controle_01 = sanitize_dataframe(controle_01, inplace=True)
controle_01 = preencher_missing_com_stats(controle_01, stats)

## Salvamento dos Dados Processados

In [11]:
# Reanexar target Para os Datasets
# Treino
abt01_test_final = test_01.copy()
abt01_test_final[TARGET] = test[TARGET].values

# Controle
abt01_controle_final = controle_01.copy()
abt01_controle_final[TARGET] = controle[TARGET].values

In [12]:
# salvar datasets test e controle tratados em parquet
TEST_FILE = PREDICTIONS_DIR / 'abt01_test.parquet'
CONTROLE_FILE = PREDICTIONS_DIR / 'abt01_controle.parquet'

print('\n💾 Salvando dados processados...')

abt01_test_final.to_parquet(TEST_FILE, index=False)
print(f"✓ Teste salvo: {TEST_FILE}")

abt01_controle_final.to_parquet(CONTROLE_FILE, index=False)
print(f"✓ Controle salvo: {CONTROLE_FILE}")

print('✅ Persistência concluída')


💾 Salvando dados processados...
✓ Teste salvo: C:\Users\billy.reis\Desktop\hackathon_dev\01_data\predictions\abt01_test.parquet
✓ Controle salvo: C:\Users\billy.reis\Desktop\hackathon_dev\01_data\predictions\abt01_controle.parquet
✅ Persistência concluída
